# Real-life photo test: does the model work outside the dataset?

**UECS2523 Applied Machine Learning, group assignment**

This notebook loads the models exported by notebook 01 (best FCNN and best CNN) and, if you ran it, notebook 03 (the **robust** CNN and FCNN trained with background swap and extra data). It runs them on photos we take ourselves. Nothing is retrained here, so it runs in a minute and works on a CPU runtime too.

| Part | What it does |
|---|---|
| 1 | Load the models and their settings from Google Drive |
| 2 | One-photo test: upload any photo and see every model's top-3 guesses, a Grad-CAM heatmap and the zoom-search result |
| 3 | Live test with the laptop webcam |
| 4 | Scored test on our own labelled photo set: real-life vs. test-set performance, by condition, and the white-background shortcut test |
| 5 | Web app (Gradio): open it on a phone, snap a photo, crop it if needed |

**Zoom search.** The models were trained on photos where the object fills most of the frame. In a real photo the object is often small or off-centre (slippers on a shelf, a can on a table). Zoom search cuts the photo into overlapping square crops at three sizes, asks the main model about each one, and keeps the crop it is most sure about. It is reported separately from the plain result, as a secondary method.

### Photo protocol for the scored test (Part 4)
Fix the list of items **before** running any model, and never retake or drop a photo after the models have seen it.
* **3 different physical items per class** (36 items), each photographed in **4 conditions**, so 12 photos per class and 144 in total:
  `whitebg` (on white paper), `darkbg` (on a dark or wooden surface), `cluttered` (normal room background), `hand` (held in a hand).
* The `whitebg` and `darkbg` shots of the same item are a controlled test of the white-background shortcut found in the EDA.
* One item per photo, filling most of the centre of the frame like the dataset images (the model centre-crops to a square).
* File names: `<class>_<condition>_<item>_<n>.jpg`, for example `plastic_darkbg_bottle1_1.jpg` or `white-glass_hand_jar2_1.jpg`.
* Label with written rules fixed before testing: clear glass = white-glass; dirty or mixed-material items = trash. Leave out items that fit no class.
* If you also add your own photos to training (notebook 03, `train_photos/`), the test photos must show **different items in different places** (other rooms, surfaces and days, ideally another group member). Otherwise the robust model could score well just because it has seen our rooms. Take this scored set **after** notebook 03's settings are final, and do not change anything because of its results.
* Any phone camera works, portrait or landscape (rotation is fixed from EXIF). iPhone `.heic` photos work too.

## 1. Load the models

In [ ]:
import os, io, json, time, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
from IPython.display import display

import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from scipy.stats import binomtest

try:  # iPhone HEIC photos: used by Part 2 uploads, Part 4 folders and the Gradio app
    from pillow_heif import register_heif_opener
except ImportError:
    os.system("pip install -q pillow-heif")
    from pillow_heif import register_heif_opener
register_heif_opener()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_NAME = "AML_Waste_Project"
RESAMPLE = getattr(Image, "Resampling", Image).LANCZOS
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd() / PROJECT_NAME
EXPORT_DIR = PROJECT_DIR / "export"
if not (EXPORT_DIR / "meta.json").exists() and (PROJECT_DIR / "quick_run" / "export" / "meta.json").exists():
    EXPORT_DIR = PROJECT_DIR / "quick_run" / "export"
    print("Only QUICK_RUN models found. Fine for testing the demo, but retrain with QUICK_RUN = False for real results.")

if not (EXPORT_DIR / "meta.json").exists():
    print(f"No exported models found in {EXPORT_DIR}.")
    if IN_COLAB:
        print("Upload 'waste_models_export.zip' (created at the end of notebook 01 or 03):")
        from google.colab import files
        up = files.upload()
        EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        for fname, data in up.items():
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                z.extractall(EXPORT_DIR)
    else:
        raise FileNotFoundError("Run notebook 01 first, or copy its export folder here.")

META = json.loads((EXPORT_DIR / "meta.json").read_text())
CLASS_NAMES = META["class_names"]
IMG_SIZE = META["img_size"]
BINS = META.get("bins", {})

# display name -> key in meta.json; the robust models exist only after notebook 03
MODEL_KEYS = {"CNN-robust": "cnn_robust", "CNN-robust-noown": "cnn_robust_noown", "CNN": "cnn", "FCNN": "fcnn",
              "FCNN-robust": "fcnn_robust"}
MODELS, LAYERS = {}, {}
for name, key in MODEL_KEYS.items():
    info = META["models"].get(key)
    if info is None:
        continue
    m = keras.models.load_model(EXPORT_DIR / info["file"], compile=False)
    m.compile(jit_compile=False)
    MODELS[name] = m
    if name.startswith("CNN"):
        LAYERS[name] = info.get("gradcam_layer") or META["gradcam_layer"]

PRIMARY = "CNN-robust" if "CNN-robust" in MODELS else "CNN"      # the model whose answer the demo shows first
UNSURE_BELOW = META.get("unsure_below_robust", 0.5) if PRIMARY == "CNN-robust" else META.get("unsure_below", 0.5)
cnn_main = MODELS[PRIMARY]

print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"Models loaded: {', '.join(MODELS)} | main model: {PRIMARY} | 'not sure' below {UNSURE_BELOW:.2f} (chosen on validation)")
display(pd.DataFrame({n: META["models"][MODEL_KEYS[n]] for n in MODELS}).T)
if META.get("quick_run"):
    print("WARNING: these models come from a QUICK_RUN (2 epochs). Retrain with QUICK_RUN = False for real results.")
if PRIMARY != "CNN-robust":
    print("Tip: run notebook 03 to add the robust CNN trained with background swap and extra data.")

### Same preprocessing as in training
A photo goes through exactly the steps the training images went through: fix rotation, convert to RGB, centre square crop, resize to 128 × 128. The models standardise pixel values themselves.

In [ ]:
def to_rgb(im):
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.getchannel("A"))
        return bg
    return im.convert("RGB")


def center_crop_resize(im, size):
    w, h = im.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return im.crop((left, top, left + s, top + s)).resize((size, size), RESAMPLE)


def open_photo(src):
    """src = file path, raw bytes, a PIL image or a Gradio ImageEditor value. Returns an upright RGB PIL image."""
    if isinstance(src, dict):                       # gr.ImageEditor: use the cropped composite
        comp = src.get("composite")
        src = comp if comp is not None else src.get("background")
    if isinstance(src, (bytes, bytearray)):
        im = Image.open(io.BytesIO(src))
    elif isinstance(src, Image.Image):
        im = src
    elif isinstance(src, np.ndarray):
        im = Image.fromarray(src)
    else:
        im = Image.open(src)
    return to_rgb(ImageOps.exif_transpose(im))


def load_photo(src):
    """Returns (original PIL image, model input uint8 array)."""
    im = open_photo(src)
    return im, np.asarray(center_crop_resize(im, IMG_SIZE), dtype=np.uint8)


# Where each class goes in Malaysia's three-colour recycling bins (blue = paper, brown = glass,
# orange = plastic / aluminium / metal). Classes not listed simply show no tip.
DISPOSAL = {
    "paper": "Blue recycling bin (paper)", "cardboard": "Blue recycling bin (paper)",
    "brown-glass": "Brown recycling bin (glass)", "green-glass": "Brown recycling bin (glass)",
    "white-glass": "Brown recycling bin (glass)", "glass": "Brown recycling bin (glass)",
    "plastic": "Orange recycling bin (plastic, aluminium, metal)",
    "metal": "Orange recycling bin (plastic, aluminium, metal)",
    "battery": "Not a household bin: e-waste / battery collection point",
    "biological": "Food or garden waste (compost), not a recycling bin",
    "clothes": "Donate or textile recycling bin", "shoes": "Donate or textile recycling bin",
    "trash": "General waste",
}

In [ ]:
def predict(model, arr, tta=False):
    """Class probabilities for one 0-255 uint8 image. tta=True also averages a mirrored copy and a tighter crop."""
    batch = [arr]
    if tta:
        batch.append(arr[:, ::-1])
        m = IMG_SIZE // 10
        batch.append(np.asarray(Image.fromarray(arr[m:-m, m:-m]).resize((IMG_SIZE, IMG_SIZE), RESAMPLE)))
    x = tf.convert_to_tensor(np.stack(batch).astype("float32"))
    return model(x, training=False).numpy().mean(axis=0)


def zoom_boxes(w, h, fracs=(1.0, 0.75, 0.55)):
    """Overlapping square crops (left, top, right, bottom) at three sizes, sliding along both axes.
    The first box is the plain centre crop, so zoom search can always fall back to the normal view."""
    s0 = min(w, h)
    boxes = [((w - s0) // 2, (h - s0) // 2, (w - s0) // 2 + s0, (h - s0) // 2 + s0)]
    for f in fracs:
        side = int(round(s0 * f))
        nx = 1 + int(np.ceil((w - side) / (side / 2))) if w > side else 1
        ny = 1 + int(np.ceil((h - side) / (side / 2))) if h > side else 1
        for x in np.linspace(0, w - side, nx):
            for y in np.linspace(0, h - side, ny):
                b = (int(x), int(y), int(x) + side, int(y) + side)
                if b not in boxes:
                    boxes.append(b)
    return boxes


def zoom_search(model, im):
    """Predict every crop and keep the one the model is most sure about. Returns (probs, box in the original
    photo's pixels, crop array). Its confidence is a maximum over ~20 crops, so it is NOT calibrated."""
    f = max(1, min(im.size) // 1024)
    small = im.reduce(f) if f > 1 else im              # 12 MP phone photos: crop a smaller copy, much faster
    boxes = zoom_boxes(*small.size)
    crops = np.stack([np.asarray(small.crop(b).resize((IMG_SIZE, IMG_SIZE), RESAMPLE), dtype=np.uint8) for b in boxes])
    probs = model.predict(crops.astype("float32"), verbose=0, batch_size=64)
    k = int(probs.max(1).argmax())
    return probs[k], tuple(v * f for v in boxes[k]), crops[k]


def grad_cam(model, arr, layer_name, class_idx=None):
    out = model.get_layer("class_probs")      # rebuild the pre-softmax class score (Selvaraju et al., 2017)
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.get_layer("head_relu").output])
    x = tf.convert_to_tensor(arr[None].astype("float32"))
    with tf.GradientTape() as tape:
        fmap, h = grad_model(x, training=False)
        logits = tf.matmul(tf.cast(h, tf.float32), tf.convert_to_tensor(out.kernel)) + tf.convert_to_tensor(out.bias)
        if class_idx is None:
            class_idx = int(tf.argmax(logits[0]))
        score = logits[:, class_idx]
    grads = tf.cast(tape.gradient(score, fmap), tf.float32)
    fmap = tf.cast(fmap, tf.float32)
    cam = tf.nn.relu(tf.reduce_sum(fmap * tf.reduce_mean(grads, axis=(1, 2))[:, None, None, :], axis=-1))[0].numpy()
    return cam / (cam.max() + 1e-8)


def overlay(arr, cam, alpha=0.45):
    heat = np.asarray(Image.fromarray(np.uint8(cam * 255)).resize((arr.shape[1], arr.shape[0]), Image.BILINEAR)) / 255.0
    return np.clip((1 - alpha) * arr / 255.0 + alpha * plt.get_cmap("jet")(heat)[..., :3], 0, 1)


def verdict(probs, threshold=None):
    threshold = UNSURE_BELOW if threshold is None else threshold
    k = int(np.argmax(probs))
    return CLASS_NAMES[k] if probs[k] >= threshold else f"not sure (maybe {CLASS_NAMES[k]})"


COLORS = {"CNN-robust": "#2E7D32", "CNN": "#4C78A8", "FCNN": "#E45756", "FCNN-robust": "#F58518"}


def show_prediction(src, title=None, tta=False, zoom=True):
    original, arr = load_photo(src)
    probs = {n: predict(m, arr, tta) for n, m in MODELS.items()}
    cam = grad_cam(cnn_main, arr, LAYERS[PRIMARY])
    z = zoom_search(cnn_main, original) if zoom else None

    n_bar = len(probs)
    fig = plt.figure(figsize=(4.2 * (3 + n_bar) / 1.4, 3.8))
    gs = fig.add_gridspec(1, 3 + n_bar, width_ratios=[1.2, 1, 1] + [1.25] * n_bar)
    ax = fig.add_subplot(gs[0]); ax.imshow(original); ax.axis("off"); ax.set_title(title or "your photo", fontsize=9)
    if z is not None:
        l, t, r, b = z[1]
        ax.add_patch(patches.Rectangle((l, t), r - l, b - t, fill=False, ec="#2E7D32", lw=2.5))
    ax = fig.add_subplot(gs[1]); ax.imshow(arr); ax.axis("off"); ax.set_title(f"model input {IMG_SIZE}x{IMG_SIZE}", fontsize=9)
    ax = fig.add_subplot(gs[2]); ax.imshow(overlay(arr, cam)); ax.axis("off"); ax.set_title(f"{PRIMARY} Grad-CAM", fontsize=9)
    for k, (n, p) in enumerate(probs.items()):
        ax = fig.add_subplot(gs[3 + k])
        top = np.argsort(p)[::-1][:3]
        ax.barh([CLASS_NAMES[i] for i in top][::-1], p[top][::-1], color=COLORS.get(n, "#888888"))
        ax.set_xlim(0, 1); ax.set_title(f"{n}: {verdict(p) if n == PRIMARY else CLASS_NAMES[int(p.argmax())]}", fontsize=9)
        for y, v in enumerate(p[top][::-1]):
            if v > 0.75:
                ax.text(v - 0.02, y, f"{v:.0%}", va="center", ha="right", fontsize=8, color="white", fontweight="bold")
            else:
                ax.text(v + 0.02, y, f"{v:.0%}", va="center", fontsize=8)
    lines = []
    best = CLASS_NAMES[int(np.argmax(probs[PRIMARY]))]
    if probs[PRIMARY].max() >= UNSURE_BELOW and best in DISPOSAL:
        lines.append(f"{PRIMARY} says {best}  →  {DISPOSAL[best]}")
    if z is not None:
        zb = CLASS_NAMES[int(z[0].argmax())]
        lines.append(f"Zoom search (green box): {zb} ({z[0].max():.0%})")
    if lines:
        fig.text(0.5, -0.06, "     |     ".join(lines), ha="center", fontsize=10)
    plt.tight_layout(); plt.show()
    return probs, z

## 2. One-photo test: upload any photo
Run the cell, click **Choose Files** and pick one or more photos from your phone or laptop.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        try:
            show_prediction(data, title=fname)
        except Exception as e:
            print(f"Could not read {fname}: {type(e).__name__}: {e}")
else:
    print("Not in Colab: call show_prediction('path/to/photo.jpg') instead.")

## 3. Live test with the webcam
Run the cell, allow camera access, hold an item in front of the camera and press **Capture**. Run it again for the next item.

In [ ]:
def take_photo(quality=0.9):
    from IPython.display import Javascript
    from google.colab.output import eval_js
    from base64 import b64decode
    display(Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      capture.style.cssText = 'font-size:16px;padding:6px 18px;margin:6px 0;';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }"""))
    data = eval_js(f"takePhoto({quality})")
    return b64decode(data.split(",")[1])


if IN_COLAB:
    try:
        show_prediction(take_photo(), title="webcam photo")
    except Exception as e:
        print("Webcam not available here (", e, "). Use Part 2 or Part 5 instead.")

## 4. Scored test on our own labelled photos

Put the photos in Google Drive, one folder per class, using **exactly** the class names below:
```
MyDrive/AML_Waste_Project/real_photos/
    battery/      battery_whitebg_aa1_1.jpg, battery_darkbg_aa1_1.jpg, battery_hand_9v_1.jpg, ...
    cardboard/    ...
```
The next cell creates the empty folders for you. Upload the photos into them (drag and drop in drive.google.com), then run the cell after it.

This gives the number the lecturer asked about: how well the models do on our own photos compared with the dataset's test set. The gap measures **domain shift**, meaning how different real photos are from the training pictures.

Two details keep the comparison honest:
* The dataset test set is imbalanced (about a third clothes) while our photo set is balanced, so both sides are compared on **balanced accuracy** and **macro F1**, not plain accuracy.
* Photos of the same item are not independent, so the 95% intervals resample **whole items**, not single photos.

Rows marked *(secondary)* use test-time augmentation or zoom search. They are extra information; the plain rows are the headline result.

In [ ]:
REAL_DIR = PROJECT_DIR / "real_photos"
for c in CLASS_NAMES:
    (REAL_DIR / c).mkdir(parents=True, exist_ok=True)
print("Put your labelled photos in:", REAL_DIR)
print("Sub-folders:", ", ".join(CLASS_NAMES))

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic", ".heif"}
CONDITIONS = ("whitebg", "darkbg", "cluttered", "hand")
USE_TTA = True        # also score test-time augmentation (secondary)
USE_ZOOM = True       # also score zoom search with the main model (secondary)

rows, arrays, bad_names = [], [], []
for k, c in enumerate(CLASS_NAMES):
    for p in sorted((REAL_DIR / c).glob("*")):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        try:
            original, arr = load_photo(p)
        except Exception as e:
            print("skip", p.name, e)
            continue
        tokens = p.stem.lower().split("_")                 # <class>_<condition>_<item>[_<n>]
        cond = next((t for t in CONDITIONS if t in tokens), "unspecified")
        item = tokens[2] if len(tokens) >= 3 else p.stem
        if len(tokens) < 3 or cond == "unspecified":
            bad_names.append(p.name)
        row = {"file": p.name, "true": c, "y": k, "condition": cond, "item": item}
        for n, m in MODELS.items():
            pr = predict(m, arr)
            row[f"{n}_pred"], row[f"{n}_conf"] = CLASS_NAMES[int(pr.argmax())], float(pr.max())
            if USE_TTA:
                row[f"{n}+TTA_pred"] = CLASS_NAMES[int(predict(m, arr, tta=True).argmax())]
        if USE_ZOOM:
            for n in [m for m in dict.fromkeys(("CNN", PRIMARY)) if m in MODELS]:
                zp, _, _ = zoom_search(MODELS[n], original)
                row[f"{n}+zoom_pred"] = CLASS_NAMES[int(zp.argmax())]
        rows.append(row); arrays.append(arr)

real = pd.DataFrame(rows)
print(f"{len(real)} labelled real-life photos found")
if bad_names:
    print(f"{len(bad_names)} file names do not follow <class>_<condition>_<item>_<n> (items/conditions may be wrong): "
          + ", ".join(bad_names[:10]) + (" ..." if len(bad_names) > 10 else ""))
if len(real):
    display(real["true"].value_counts().rename("photos").to_frame().T)

In [ ]:
def wilson_ci(k, n, z=1.96):
    """95% Wilson interval for an accuracy of k correct out of n."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half


def boot_ci(df, pred_col, fn, n_boot=2000, seed=0):
    """95% interval that resamples whole items within each class (photos of one item are not independent)."""
    r = np.random.default_rng(seed)
    groups = {c: [g for _, g in d.groupby("item")] for c, d in df.groupby("true")}
    vals = []
    for _ in range(n_boot):
        sample = pd.concat([gs[i] for gs in groups.values() for i in r.integers(0, len(gs), len(gs))])
        vals.append(fn(sample["true"], sample[pred_col]))
    return np.percentile(vals, [2.5, 97.5])


if len(real):
    present = sorted(real["true"].unique())
    missing = [c for c in CLASS_NAMES if c not in present]
    if missing:
        print(f"No real photos yet for: {', '.join(missing)}. Real-life scores cover the {len(present)} classes photographed.")
    mf1 = lambda t, p_: f1_score(t, p_, labels=present, average="macro", zero_division=0)
    for n in MODELS:
        real[f"{n}_correct"] = real[f"{n}_pred"] == real["true"]

    variants = list(MODELS) + [f"{n}+TTA" for n in MODELS] + [f"{n}+zoom" for n in dict.fromkeys(("CNN", PRIMARY)) if n in MODELS]
    rows_s = []
    for v in variants:
        if f"{v}_pred" not in real:
            continue
        mm = META["models"][MODEL_KEYS[v.split("+")[0]]]
        lo, hi = boot_ci(real, f"{v}_pred", balanced_accuracy_score)
        rows_s.append({"model": v + (" (secondary)" if "+" in v else ""),
                       "test-set balanced acc": mm.get("test_balanced_acc", np.nan),
                       "real-life balanced acc": balanced_accuracy_score(real["true"], real[f"{v}_pred"]),
                       "95% CI (items resampled)": f"{lo:.2f} to {hi:.2f}",
                       "test-set macro F1": mm.get("test_macro_f1", np.nan),
                       "real-life macro F1": mf1(real["true"], real[f"{v}_pred"]),
                       "real-life bin accuracy": float(np.mean(real["true"].map(BINS) == real[f"{v}_pred"].map(BINS))) if BINS else np.nan})
    summary = pd.DataFrame(rows_s).set_index("model")
    summary["gap (balanced acc)"] = summary["test-set balanced acc"] - summary["real-life balanced acc"]
    summary.to_csv(PROJECT_DIR / "real_life_summary.csv")
    display(summary.style.format({c: "{:.3f}" for c in summary.columns if c != "95% CI (items resampled)"}, na_rep="-"))
    print(f"{len(real)} photos of {real.groupby(['true', 'item']).ngroups} items. If the test-set figure lies above the "
          "interval, the drop on real photos is larger than chance alone would explain.")

    def paired_item_boot(df, a, b, fn=balanced_accuracy_score, n_boot=2000, seed=0):
        """fn(b) - fn(a) on the same photos; 95% CI resampling whole items within each class."""
        df = df.reset_index(drop=True)
        r = np.random.default_rng(seed)
        groups = [[g.index.values for _, g in d.groupby("item")] for _, d in df.groupby("true")]
        t, pa, pb = df["true"].values, df[f"{a}_pred"].values, df[f"{b}_pred"].values
        diffs = [fn(t[i], pb[i]) - fn(t[i], pa[i]) for i in
                 (np.concatenate([gs[k] for gs in groups for k in r.integers(0, len(gs), len(gs))]) for _ in range(n_boot))]
        return fn(t, pb) - fn(t, pa), *np.percentile(diffs, [2.5, 97.5])

    def item_sign_test(df, a, b):
        per = df.groupby(["true", "item"]).agg(a=(f"{a}_correct", "mean"), b=(f"{b}_correct", "mean"))
        up, down = int((per.b > per.a).sum()), int((per.b < per.a).sum())
        return up, down, (binomtest(min(up, down), up + down, 0.5).pvalue if up + down else 1.0)

    print("Paired comparisons on the same photos (primary result: the item-level interval, not single photos):")
    for a, b in [("CNN", "CNN-robust-noown"), ("CNN", "CNN-robust"), ("FCNN", "CNN")]:
        if a in MODELS and b in MODELS:
            d, lo, hi = paired_item_boot(real, a, b)
            up, down, p_s = item_sign_test(real, a, b)
            print(f"  {b} - {a}: balanced accuracy {d:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}, items resampled); "
                  f"items better {up}, worse {down}, item sign test p = {p_s:.3f}")

    kept = real[f"{PRIMARY}_conf"] >= UNSURE_BELOW
    print(f"With the 'not sure' threshold {UNSURE_BELOW:.2f} (chosen on validation) {PRIMARY} answers {kept.mean():.0%} of photos "
          f"and is right on {real.loc[kept, f'{PRIMARY}_correct'].mean() if kept.any() else float('nan'):.0%} of those.")
    print(f"Mean {PRIMARY} confidence {real[f'{PRIMARY}_conf'].mean():.0%} vs actual accuracy {real[f'{PRIMARY}_correct'].mean():.0%} "
          "(confidence far above accuracy = over-confident on new kinds of photo).")

    by_cond = real.groupby("condition").agg(photos=("file", "size"), **{f"{n} acc": (f"{n}_correct", "mean") for n in MODELS})
    by_cond[f"{PRIMARY} 95% CI"] = ["%.2f to %.2f" % wilson_ci(int(round(a * n)), int(n))
                                    for a, n in zip(by_cond[f"{PRIMARY} acc"], by_cond["photos"])]
    print("Accuracy by photo condition:")
    display(by_cond.style.format({"photos": "{:.0f}", **{f"{n} acc": "{:.0%}" for n in MODELS}}))

    pair = real[real["condition"].isin(["whitebg", "darkbg"])]
    if pair["condition"].nunique() == 2:
        print("\nWhite-background shortcut test (the same items on white paper vs a dark surface):")
        suspects = [c for c in ("battery", "trash") if c in CLASS_NAMES]
        for n in MODELS:
            t = pair.assign(ok=pair[f"{n}_correct"].astype(float)).pivot_table(index=["true", "item"], columns="condition",
                                                                              values="ok", aggfunc="mean").dropna()
            wb, db = pair[pair["condition"] == "whitebg"], pair[pair["condition"] == "darkbg"]
            fa = wb.loc[~wb["true"].isin(suspects), f"{n}_pred"].isin(suspects).mean()
            fb = db.loc[~db["true"].isin(suspects), f"{n}_pred"].isin(suspects).mean()
            print(f"  {n}: accuracy {t['whitebg'].mean():.0%} on white vs {t['darkbg'].mean():.0%} on dark ({len(t)} items); "
                  f"other items called battery/trash: {fa:.0%} on white vs {fb:.0%} on dark")

    by_class = real.groupby("true").agg(**{n: (f"{n}_correct", "mean") for n in MODELS})
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    by_class.plot.bar(ax=axes[0], color=[COLORS.get(n, "#888888") for n in MODELS], ylim=(0, 1.12), rot=45)
    axes[0].legend(ncol=len(MODELS), loc="upper right", frameon=False)
    axes[0].set(title="Real-life accuracy per class", ylabel="accuracy", xlabel="")
    cm = confusion_matrix(real["true"], real[f"{PRIMARY}_pred"], labels=CLASS_NAMES)
    axes[1].imshow(cm, cmap="Blues")
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            if cm[i, j]:
                axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
    axes[1].set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=60, ha="right", fontsize=8)
    axes[1].set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=8)
    axes[1].set(title=f"{PRIMARY} confusion matrix on our real-life photos (counts)", xlabel="predicted", ylabel="true")
    plt.tight_layout()
    fig.savefig(PROJECT_DIR / "figures_real_life_summary.png", bbox_inches="tight", dpi=150)
    plt.show()
    real.to_csv(PROJECT_DIR / "real_life_predictions.csv", index=False)
else:
    print("No photos yet: add them to the folders above and run the previous cell again.")

In [ ]:
if len(real):
    cols = 8
    n = len(real)
    fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(2 * cols, 2.3 * int(np.ceil(n / cols))), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r), arr in zip(axes.flat, real.iterrows(), arrays):
        ax.imshow(arr)
        ok = r[f"{PRIMARY}_pred"] == r["true"]
        ax.set_title(f"{r['true']}\n{PRIMARY}: {r[f'{PRIMARY}_pred']} {r[f'{PRIMARY}_conf']:.0%}", fontsize=7,
                     color="green" if ok else "crimson")
    fig.suptitle(f"Every real-life photo (green = {PRIMARY} correct, red = wrong)")
    plt.tight_layout()
    fig.savefig(PROJECT_DIR / "figures_real_life_gallery.png", bbox_inches="tight", dpi=150)
    plt.show()

    wrong = list(real.index[real[f"{PRIMARY}_pred"] != real["true"]])
    if wrong:
        print(f"Grad-CAM on all {len(wrong)} {PRIMARY} mistakes: is it looking at the item or the background?")
        per_row = 8
        n_rows = int(np.ceil(len(wrong) / per_row))
        fig, axes = plt.subplots(2 * n_rows, per_row, figsize=(2.2 * per_row, 4.6 * n_rows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for j, i in enumerate(wrong):
            r, col = 2 * (j // per_row), j % per_row
            axes[r, col].imshow(arrays[i])
            axes[r, col].set_title(f"true: {real.loc[i, 'true']}\n{real.loc[i, 'condition']}", fontsize=7)
            axes[r + 1, col].imshow(overlay(arrays[i], grad_cam(cnn_main, arrays[i], LAYERS[PRIMARY])))
            axes[r + 1, col].set_title(f"pred: {real.loc[i, f'{PRIMARY}_pred']}", fontsize=7, color="crimson")
        plt.tight_layout()
        fig.savefig(PROJECT_DIR / "figures_real_life_mistakes_gradcam.png", bbox_inches="tight", dpi=150)
        plt.show()

## 5. Web app: test it from a phone

This starts a small Gradio web page (Gradio comes preinstalled on Colab). Colab prints a public link (`https://….gradio.live`). It works while this cell keeps running. Open it on any phone, take or upload a photo, **use the crop tool to draw a square around the item if it is small**, and press Submit. Tick "Zoom search" to let the model look for the item itself.

The link is public: anyone who has it can use the demo until you stop the cell. Stop the app with the stop button of this cell when you are done.

In [ ]:
try:
    import gradio as gr
except ImportError:
    os.system("pip install -q gradio")
    import gradio as gr
print("Gradio", gr.__version__)


def to_label(p):
    return {c: float(v) for c, v in zip(CLASS_NAMES, p)}


def app_predict(img, use_zoom):
    if img is None or (isinstance(img, dict) and img.get("composite") is None and img.get("background") is None):
        return [None] * len(MODELS) + [None, ""]
    im = open_photo(img)
    arr = np.asarray(center_crop_resize(im, IMG_SIZE), dtype=np.uint8)
    note, zoomed = "", False
    if use_zoom:
        zp, box, zarr = zoom_search(cnn_main, im)
        if zp.max() > predict(cnn_main, arr).max():
            arr, zoomed = zarr, True
            note = " (zoom search picked the most confident crop; its confidence is not calibrated)"
    outs = [to_label(predict(m, arr)) for m in MODELS.values()]
    p_main = predict(cnn_main, arr)
    heat = np.uint8(overlay(arr, grad_cam(cnn_main, arr, LAYERS[PRIMARY])) * 255)
    best = CLASS_NAMES[int(p_main.argmax())]
    if p_main.max() < UNSURE_BELOW and not zoomed:
        tip = f"Not sure. Best guess: {best} ({p_main.max():.0%}). Crop closer to the item or try a plainer background."
    else:
        tip = f"{best} ({p_main.max():.0%})" + (f": {DISPOSAL[best]}" if best in DISPOSAL else "")
    return outs + [heat, tip + note]


try:   # an image box with a crop tool (Gradio 4+)
    image_in = gr.ImageEditor(type="pil", sources=["upload", "webcam"], transforms=("crop",), layers=False,
                              brush=False, eraser=False, label="Photo of one waste item (crop around it if it is small)")
except TypeError:
    image_in = gr.Image(type="pil", sources=["upload", "webcam"], label="Photo of one waste item")

GR_MAJOR = int(gr.__version__.split(".")[0])
app_kwargs = dict(
    fn=app_predict,
    inputs=[image_in, gr.Checkbox(value=True, label="Zoom search (find the item in a big photo)")],
    outputs=[gr.Label(num_top_classes=3, label=f"{n} ({META['models'][MODEL_KEYS[n]]['run']}, trained from scratch)")
             for n in MODELS] + [gr.Image(label=f"Where {PRIMARY} looked (Grad-CAM)"),
                                 gr.Textbox(label="Verdict and disposal tip")],
    title="Household waste classifier: FCNN vs CNN trained from scratch",
    description="UECS2523 Applied Machine Learning. Take or upload a photo of ONE item "
                "(bottle, can, box, paper, clothes, shoes, battery, glass jar, food waste...).",
)
app_kwargs["flagging_mode" if GR_MAJOR >= 5 else "allow_flagging"] = "never"
app = gr.Interface(**app_kwargs)

LAUNCH_APP = True
if LAUNCH_APP:
    # debug=True keeps this cell running (and the link alive) until you press stop
    app.launch(share=IN_COLAB, debug=IN_COLAB)